# Feature engineering
In this step we will introduce features learned from the data exploration step, re-formatting the information we have into a language the model can understand. <br>
This includes: <br>
- splitting wind speed into discrete bins to approach the "S" curve <br>
- splitting wind directions into discrete bins (NE, NW, SW, SE) <br>
- create cycling time feature (hour_of_day, month_of_year)


## Import necessary libraries

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

## Load data

In [42]:
path1 = "../Data/Processed/wind_persistance_polar.csv"
path2 = "../Data/Processed/wind_average_polar.csv"
path3 = "../Data/Processed/wind_hourly_polar.csv"

def load_data(path):
    data = pd.read_csv(path)
    return data

wind_persistance_polar = load_data(path1)
wind_average_polar = load_data(path2)
wind_hourly_polar = load_data(path3)

wind_persistance_polar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27048 entries, 0 to 27047
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   TIMESTAMP            27048 non-null  object 
 1   POWER                27048 non-null  float64
 2   wind_speed_10m       27048 non-null  float64
 3   wind_direction_10m   27048 non-null  float64
 4   wind_speed_100m      27048 non-null  float64
 5   wind_direction_100m  27048 non-null  float64
dtypes: float64(5), object(1)
memory usage: 1.2+ MB


Somehow the timestamp is not datetime type 

## Create features

In [43]:
def create_features(wind):
    """
    Create features to improve wind power forecasting.
    """
    wind_feat = wind.copy()
    wind_feat['TIMESTAMP'] = pd.to_datetime(wind_feat['TIMESTAMP']) 

    # 1. Wind speed bins using 100m (linearizing S-curve)
    # Based on the plots, we define three regions:
    # a. Cut-in: 0-4 m/s
    # b. Ramp-up: 4-20 m/s
    # c. Cut-out: >20 m/s

    speed_bins = [0, 4, 20, 50]
    speed_labels = ['cut-in', 'ramp-up', 'cut-out']

    wind_feat['wind_speed_bin'] = pd.cut(wind_feat['wind_speed_100m'], bins=speed_bins, labels=speed_labels, right=False, include_lowest=True)

    # 2. Wind direction bins
    dir_bins = [0, 90, 180, 270, 360]
    dir_labels = ['NE', 'SE', 'SW', 'NW']
    wind_feat['wind_dir_bin'] = pd.cut(wind_feat['wind_direction_100m'], bins=dir_bins, labels=dir_labels, right=False, include_lowest=True)

    # 3. Cyclical Time Features (hour)
    wind_feat['hour'] = wind_feat['TIMESTAMP'].dt.hour
    wind_feat['hour_sin'] = np.sin(2 * np.pi * wind_feat['hour'] / 24)
    wind_feat['hour_cos'] = np.cos(2 * np.pi * wind_feat['hour'] / 24)

    # 4. Cyclical Time Features (month)
    wind_feat['month'] = wind_feat['TIMESTAMP'].dt.month
    wind_feat['month_sin'] = np.sin(2 * np.pi * wind_feat['month'] / 12)
    wind_feat['month_cos'] = np.cos(2 * np.pi * wind_feat['month'] / 12)

    return wind_feat


### Apply features to all datasets

In [44]:
# List of features to use
features_to_use = [
    'POWER',
    'wind_speed_100m',
    'wind_speed_bin',
    'wind_dir_bin',
    'hour_sin',
    'hour_cos',
    'month_sin',
    'month_cos'
]

# Create features for persistence data
wind_pers_feat = create_features(wind_persistance_polar)
wind_pers_feat = wind_pers_feat[features_to_use].copy()

# Create features for average data
wind_avg_feat = create_features(wind_average_polar)
wind_avg_feat = wind_avg_feat[features_to_use].copy()

# Create features for hourly data
wind_hour_feat = create_features(wind_hourly_polar)
wind_hour_feat = wind_hour_feat[features_to_use].copy()

print("\nDataFrames are ready for preprocessing.")


DataFrames are ready for preprocessing.


## Create preprocessing pipeline

In [45]:
def get_preprocessor():
    """
    Define and return preprocessing pipeline.
    """
    # 1. Define feature lists
    numerical_features = ['wind_speed_100m']
    categorical_features = ['wind_speed_bin', 'wind_dir_bin']
    passthrough_features = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos']

    # 2. Define the transformers
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    # 3. Create the ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical_features),
            ('cat', categorical_transformer, categorical_features),
            ('pass', 'passthrough', passthrough_features)
        ]
    )

    return preprocessor


### Apply preprocessing to all datasets

Split feature data into X (input data) and y (output data). <br>
Fit the preprocessing pipeline to all three datasets.

In [46]:
preprocessor = get_preprocessor()

# Apply preprocessing to persistance data
# 1. Split into X and y
X_pers = wind_pers_feat.drop('POWER', axis=1)
y_pers_train = wind_pers_feat['POWER']

# 2. Fit preprocessing on training data
X_pers_train = preprocessor.fit_transform(X_pers)
print("\nTransformed training data shape (persistence data):", X_pers_train.shape)

# Apply preprocessing for average data
# 1. Split into X and y
X_avg = wind_avg_feat.drop('POWER', axis=1)
y_avg_train = wind_avg_feat['POWER']

# 2. Fit preprocessing on training data
X_avg_train = preprocessor.fit_transform(X_avg)
print("\nTransformed training data shape (average data):", X_avg_train.shape)

# Apply preprocessing for hourly data
X_hour = wind_hour_feat.drop('POWER', axis=1)
y_hour_train = wind_hour_feat['POWER']

# 2. Fit preprocessing on training data
X_hour_train = preprocessor.fit_transform(X_hour)
print("\nTransformed training data shape (hourly data):", X_hour_train.shape)


Transformed training data shape (persistence data): (27048, 12)

Transformed training data shape (average data): (27048, 12)

Transformed training data shape (hourly data): (27048, 12)


### Exporting transformed data